# 03 — Cosine similarity and cell-type assignment

Compares each segmented cell's gene expression profile against the scRNA-seq prototypes
using cosine similarity. The prototype with the highest score determines the cell type.

**Inputs** (all produced by notebooks 01 and 02)
- `data/prototypes_normalized.csv`
- `data/cell_expression_aligned.csv`
- `data/shared_genes.txt`

**Outputs**
- `data/cosine_similarity_matrix.csv` — cells × cell_types similarity scores
- `data/cell_type_predictions.csv` — predicted type and best score per cell
- `outputs/similarity_histogram.png`
- `outputs/similarity_heatmap.png`
- `outputs/cell_type_counts.png`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

## Config

In [ ]:
PROTOTYPES_NORM_PATH = "../data/prototypes_normalized.csv"
CELL_EXPR_PATH       = "../data/cell_expression_aligned.csv"
SHARED_GENES_PATH    = "../data/shared_genes.txt"

SIM_MATRIX_PATH      = "../data/cosine_similarity_matrix.csv"
PREDICTIONS_PATH     = "../data/cell_type_predictions.csv"
OUTPUT_DIR           = "../outputs"

HEATMAP_N_CELLS      = 50   # number of cells to show in heatmap

## 1 · Load inputs and align genes

We use `shared_genes.txt` as the single source of truth for gene ordering.
This guarantees that prototype columns and cell columns are aligned before any comparison.

In [ ]:
with open(SHARED_GENES_PATH) as f:
    shared_genes = [line.strip() for line in f if line.strip()]

prototypes = pd.read_csv(PROTOTYPES_NORM_PATH, index_col=0)
cells      = pd.read_csv(CELL_EXPR_PATH, index_col=0)

print("Prototypes (raw):", prototypes.shape)
print("Cells (raw):",      cells.shape)

# restrict both to shared genes in the same order
prototypes = prototypes.reindex(columns=shared_genes, fill_value=0)
cells      = cells.reindex(columns=shared_genes, fill_value=0)

assert list(prototypes.columns) == list(cells.columns), "Gene order mismatch after reindex — this should not happen."
print(f"\nAligned on {len(shared_genes)} shared genes.")
print("Prototypes:", prototypes.shape, "  Cells:", cells.shape)

## 2 · L2-normalise cell expression vectors

In [ ]:
cells_norm = pd.DataFrame(
    normalize(cells.values, norm="l2"),
    index=cells.index,
    columns=cells.columns
)

# cells with zero counts across all genes can't be compared — drop them and warn
zero_cells = (cells_norm.sum(axis=1) == 0)
if zero_cells.any():
    print(f"Warning: {zero_cells.sum()} cells have zero counts and will be excluded.")
    cells_norm = cells_norm[~zero_cells]

## 3 · Compute cosine similarity

In [ ]:
sim = cosine_similarity(cells_norm.values, prototypes.values)
sim_df = pd.DataFrame(
    sim,
    index=cells_norm.index,
    columns=prototypes.index
)
print("Similarity matrix:", sim_df.shape, "(cells × cell_types)")
print(sim_df.head(3))

## 4 · Assign cell types

In [ ]:
results = pd.DataFrame({
    "predicted_cell_type": sim_df.idxmax(axis=1),
    "best_similarity":     sim_df.max(axis=1)
}, index=sim_df.index)

print(results["predicted_cell_type"].value_counts())
print(f"\nMedian best similarity: {results['best_similarity'].median():.3f}")

## 5 · Save outputs

In [ ]:
sim_df.to_csv(SIM_MATRIX_PATH)
results.to_csv(PREDICTIONS_PATH)
print(f"Saved similarity matrix   → {SIM_MATRIX_PATH}")
print(f"Saved cell type predictions → {PREDICTIONS_PATH}")

## 6 · Visualisations

In [ ]:
# distribution of best similarity scores
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(results["best_similarity"], bins=30, edgecolor="white", linewidth=0.4)
ax.set_xlabel("Cosine similarity (best prototype match)")
ax.set_ylabel("Number of cells")
ax.set_title("Distribution of cell-to-prototype similarity")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/similarity_histogram.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# heatmap — cells sorted by predicted type so structure is visible
sorted_idx = results.sort_values("predicted_cell_type").index
sim_subset = sim_df.loc[sorted_idx].iloc[:HEATMAP_N_CELLS]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(sim_subset, cmap="viridis", ax=ax)
ax.set_title(f"Cosine similarity — first {HEATMAP_N_CELLS} cells (sorted by predicted type)")
ax.set_xlabel("Cell type prototype")
ax.set_ylabel("Cell")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/similarity_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# bar chart of predicted cell type counts
counts = results["predicted_cell_type"].value_counts()

fig, ax = plt.subplots(figsize=(6, 4))
counts.plot(kind="bar", ax=ax)
ax.set_xlabel("Predicted cell type")
ax.set_ylabel("Number of cells")
ax.set_title("Predicted cell type distribution")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/cell_type_counts.png", dpi=300, bbox_inches="tight")
plt.show()